# الفصل 7: بناء تطبيقات الدردشة
## بدء سريع مع واجهة برمجة تطبيقات نماذج Microsoft Foundry

تم تكييف هذا الدفتر من [مستودع عينات Azure OpenAI](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) الذي يتضمن دفاتر تصل إلى خدمات [Azure OpenAI](notebook-azure-openai.ipynb).

> **ملاحظة:** سيتم إيقاف نماذج GitHub في نهاية يوليو 2026. يستهدف هذا الدفتر الآن [نماذج Microsoft Foundry](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)، التي تقدم نفس كتالوج النماذج المجاني للتجربة وتجربة SDK لاستدلال Azure AI.


# نظرة عامة  
"نماذج اللغة الكبيرة هي دوال تقوم بتحويل النص إلى نص. عند إعطاء سلسلة نصية كمدخل، يحاول نموذج اللغة الكبير التنبؤ بالنص الذي سيأتي بعده"(1). ستقدم هذه الدفترية "البدء السريع" للمستخدمين مفاهيم عالية المستوى لنماذج اللغة الكبيرة، والمتطلبات الأساسية للحزمة للبدء مع AML، ومقدمة بسيطة لتصميم التنبيهات، والعديد من الأمثلة القصيرة لحالات استخدام مختلفة. 


## جدول المحتويات  

[نظرة عامة](#overview)  
[كيفية استخدام خدمة OpenAI](#how-to-use-openai-service)  
[1. إنشاء خدمة OpenAI الخاصة بك](#1.-creating-your-openai-service)  
[2. التثبيت](#2.-installation)    
[3. بيانات الاعتماد](#3.-credentials)  

[حالات الاستخدام](#use-cases)    
[1. تلخيص النص](#1.-summarize-text)  
[2. تصنيف النص](#2.-classify-text)  
[3. إنشاء أسماء جديدة للمنتجات](#3.-generate-new-product-names)  
[4. ضبط دقيق لمصنف](#4.fine-tune-a-classifier)  

[المراجع](#references)


### أنشئ طلبك الأول  
ستوفر هذه التمارين القصيرة مقدمة أساسية لتقديم الطلبات إلى نموذج في Microsoft Foundry Models لمهمة بسيطة "التلخيص".


**الخطوات**:  
1. قم بتثبيت مكتبة `azure-ai-inference` في بيئة بايثون الخاصة بك، إذا لم تقم بذلك بعد.  
2. حمّل مكتبات المساعدة القياسية وقم بإعداد بيانات اعتماد Microsoft Foundry Models الخاصة بك.  
3. اختر نموذجًا لمهمتك  
4. أنشئ طلبًا بسيطًا للنموذج  
5. قدّم طلبك إلى واجهة برمجة تطبيقات النموذج!


### 1. تثبيت `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. استيراد مكتبات المساعدة وإنشاء بيانات الاعتماد


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. إيجاد النموذج المناسب  
النماذج مثل GPT-4o و GPT-4o mini يمكنها فهم اللغة الطبيعية وتوليدها، وهي متوفرة في كتالوج نماذج Microsoft Foundry جنبًا إلى جنب مع نماذج من Meta و Mistral و Cohere و Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. تصميم المطالبات  

"السحر في نماذج اللغة الكبيرة هو أنه من خلال تدريبها على تقليل خطأ التنبؤ هذا عبر كميات هائلة من النصوص، تنتهي النماذج إلى تعلم مفاهيم مفيدة لهذه التنبؤات. على سبيل المثال، تتعلم مفاهيم مثل "(1):

* كيفية التهجئة
* كيفية عمل القواعد النحوية
* كيفية إعادة الصياغة
* كيفية الإجابة على الأسئلة
* كيفية إجراء محادثة
* كيفية الكتابة بعدة لغات
* كيفية البرمجة
* إلخ.

#### كيفية التحكم في نموذج لغوي كبير  
"من بين جميع المدخلات إلى نموذج لغوي كبير، النص المطلوب (prompt) هو الأكثر تأثيرًا بفارق كبير (1).

يمكن تحفيز نماذج اللغة الكبيرة لإنتاج المخرجات بعدة طرق:

تعليمات: أخبر النموذج بما تريد
إكمال: استحث النموذج على إكمال بداية ما تريد
عرض: أر النموذج ما تريد، إما عبر:
بعض الأمثلة في النص المطلوب
عدة مئات أو آلاف من الأمثلة في مجموعة بيانات تدريب دقيقة الضبط"



#### هناك ثلاث إرشادات أساسية لإنشاء المطالبات:

**أظهر وأخبر**. اجعل ما تريد واضحًا إما من خلال التعليمات أو الأمثلة أو مزيج منهما. إذا كنت تريد أن يصنف النموذج قائمة عناصر بالترتيب الأبجدي أو يصنف فقرة حسب المشاعر، أظهر له أن هذا ما تريده.

**قدّم بيانات ذات جودة**. إذا كنت تحاول بناء مصنف أو جعل النموذج يتبع نمطًا معينًا، تأكد من وجود عدد كافٍ من الأمثلة. تأكد من تدقيق أمثلتك — النموذج عادة ذكي بما يكفي ليرى الأخطاء الإملائية الأساسية ويعطيك ردًا عليها، لكنه قد يفترض أيضًا أن هذا مقصود وقد يؤثر ذلك على الرد.

**تحقق من إعداداتك.** إعدادات درجة الحرارة و top_p تتحكم في مدى حتمية النموذج في توليد الرد. إذا كنت تطلب منه ردًا حيث يوجد جواب صحيح واحد فقط، فستريد ضبط هذه الإعدادات إلى القيم الأقل. إذا كنت تبحث عن ردود أكثر تنوعًا، قد تريد ضبطها إلى القيم الأعلى. الخطأ الأكبر الذي يرتكبه الناس مع هذه الإعدادات هو افتراض أنها ضوابط "الذكاء" أو "الإبداع".


المصدر: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. قدّم! 


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### كرر نفس المكالمة، كيف تكون مقارنة النتائج؟ 


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## تلخيص النص  
#### التحدي  
تلخيص النص بإضافة 'tl;dr:' إلى نهاية مقطع نصي. لاحظ كيف يفهم النموذج كيفية أداء عدد من المهام بدون تعليمات إضافية. يمكنك تجربة مطالبات أكثر وصفية من tl;dr لتعديل سلوك النموذج وتخصيص التلخيص الذي تتلقاه(3).  

أظهرت الأعمال الحديثة مكاسب كبيرة في العديد من مهام ومعايير معالجة اللغة الطبيعية من خلال التدريب المسبق على مجموعة كبيرة من النصوص متبوعًا بالتدريب الدقيق على مهمة محددة. في حين أن هذا الأسلوب يكون عادةً غير معتمد على المهمة من حيث البنية، إلا أنه لا يزال يتطلب مجموعات بيانات تدريب دقيقة خاصة بالمهمة مكونة من آلاف أو عشرات الآلاف من الأمثلة. بالمقابل، يمكن للبشر عموماً أداء مهمة لغوية جديدة من خلال أمثلة قليلة فقط أو تعليمات بسيطة - وهو أمر لا تزال أنظمة معالجة اللغة الطبيعية الحالية تكافح لتحقيقه إلى حد كبير. هنا نظهر أن زيادة حجم نماذج اللغة يحسن بشكل كبير أداء المهام غير المعتمد على المهمة مع أمثلة قليلة، وفي بعض الأحيان يصل إلى مستويات تنافسية مع أساليب التدريب الدقيق المتقدمة السابقة.  



tl;dr  


# تمارين لعدة حالات استخدام  
1. تلخيص النص  
2. تصنيف النص  
3. إنشاء أسماء منتجات جديدة


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## تصنيف النص  
#### التحدي  
صنّف العناصر إلى الفئات المقدمة في وقت الاستنتاج. في المثال التالي، نقدم كلًا من الفئات والنص ليتم تصنيفه في الموجه (*playground_reference). 

استفسار العميل: مرحبًا، تعطل أحد مفاتيح لوحة مفاتيح الكمبيوتر المحمول خاصتي مؤخرًا وسأحتاج إلى بديل:

الفئة المصنفة:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## توليد أسماء منتجات جديدة
#### التحدي
إنشاء أسماء للمنتجات من كلمات أمثلة. هنا ندرج في المطالبة معلومات حول المنتج الذي سنولد له أسماء. كما نوفر مثالًا مشابهًا لإظهار النمط الذي نرغب في تلقيه. لقد قمنا أيضًا بضبط قيمة درجة الحرارة عالية لزيادة العشوائية والاستجابات المبتكرة أكثر.

وصف المنتج: صانع مخفوق الحليب المنزلي
كلمات البداية: سريع، صحي، مدمج.
أسماء المنتجات: HomeShaker, Fit Shaker, QuickShake, Shake Maker

وصف المنتج: زوج من الأحذية التي تناسب أي حجم قدم.
كلمات البداية: قابل للتكيف، مناسب، أومني-فيت.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# المراجع  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [بوابة Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [أفضل الممارسات لضبط نماذج GPT لتصنيف النصوص](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# للمزيد من المساعدة  
[فريق تسويق OpenAI التجاري](AzureOpenAITeam@microsoft.com) 


# المساهمون
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
